# Stock Market Forecasting with PyTorch LSTM
## Economics PBL

This model returns stock prediction with a **LSTM** neural network



In [211]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

#i do have cuda on my device but baaki logo ke liye if they dont have it, it fallbacks to cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

# style stuff
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    pass

OUT_DIR = "eco_plots"
os.makedirs(OUT_DIR, exist_ok=True)


Using compute device: cuda


### 🎯 STEP 2: Stock Picker & Training Configuration
Edit the configuration below to choose any company (e.g. **AAPL**, **META**, **NVDA**, **MSFT**, **TSLA**, **RELIANCE.NS**, etc.) and set the number of **Epochs** for model training.


In [212]:
#listed popular stocks here (tbh idt this model works on other stocks properly, getting poor rmse for them)
#this dict is mainly for the website
AVAILABLE_STOCKS = {
    "AAPL":        {"name": "Apple Inc.",                "currency": "$", "base_price": 180.0, "volatility": 0.015},
    "META":        {"name": "Meta Platforms Inc.",       "currency": "$", "base_price": 300.0, "volatility": 0.022},
    "NVDA":        {"name": "NVIDIA Corporation",        "currency": "$", "base_price": 120.0, "volatility": 0.028},
    "MSFT":        {"name": "Microsoft Corporation",     "currency": "$", "base_price": 350.0, "volatility": 0.014},
    "AMZN":        {"name": "Amazon.com Inc.",           "currency": "$", "base_price": 150.0, "volatility": 0.018},
    "GOOGL":       {"name": "Alphabet Inc. (Google)",    "currency": "$", "base_price": 170.0, "volatility": 0.016},
    "TSLA":        {"name": "Tesla Inc.",                "currency": "$", "base_price": 220.0, "volatility": 0.032},
    "RELIANCE.NS": {"name": "Reliance Industries Ltd",   "currency": "₹", "base_price": 2500.0, "volatility": 0.012},
    "TCS.NS":      {"name": "Tata Consultancy Services", "currency": "₹", "base_price": 3500.0, "volatility": 0.011},
    "INFY.NS":     {"name": "Infosys Ltd",               "currency": "₹", "base_price": 1500.0, "volatility": 0.013},
    "^NSEI":       {"name": "NIFTY 50 Index",            "currency": "₹", "base_price": 20000.0, "volatility": 0.009},
    "^GSPC":       {"name": "S&P 500 Index",             "currency": "$", "base_price": 5000.0, "volatility": 0.009},
}

#ticker
TICKER = "TSLA"               

# 2 epochs, somehow the best epoch number is 1? and more than 10 epochs are not introducing any improvement
EPOCHS = 10     

#hyperparams
LOOKBACK_WINDOW = 60          
BATCH_SIZE = 64               # claue told me to add this
LEARNING_RATE = 1e-3          # claude told me this is an optimal learning rate for adam
PATIENCE = 10                 # as tested earlier, no gain after 10 epochs
HIDDEN_DIM = 64              
NUM_LAYERS = 2              
DROPOUT = 0.2                 
SEED = 42

#for simulating stuff, added by copilot for uhhh making the project more robust
CAPITAL = 100_000            
TC_RATE = 0.001               # transaction cost (tax and all yada yada)
SCALE_W = 100_000             # dont know what this is tbh
START_DATE = "2015-01-01"     # configurable
END_DATE = "2026-10-31"
#test val
TEST_FRAC = 0.2              
VAL_FRAC = 0.15               
# ticker selector
meta = AVAILABLE_STOCKS.get(TICKER, {
    "name": TICKER,
    "currency": "₹" if TICKER.endswith((".NS", ".BO")) else "$", #since nse stocks end with .ns
    "base_price": 100.0,
    "volatility": 0.018
})
CURRENCY = meta["currency"]
COMPANY_NAME = meta["name"]

print("=" * 60)
print(f"Selected Stock   : {TICKER} ({COMPANY_NAME})")
print(f"Currency         : {CURRENCY}")
print(f"Training Epochs  : {EPOCHS}")
print(f"Lookback Window  : {LOOKBACK_WINDOW} days")
print(f"Initial Capital  : {CURRENCY}{CAPITAL:,}")
print("=" * 60)


Selected Stock   : TSLA (Tesla Inc.)
Currency         : $
Training Epochs  : 10
Lookback Window  : 60 days
Initial Capital  : $100,000


### 📥 STEP 3: Data feeding through yfinance
Included a brownian motion fallback if yfinance is unavailable


In [213]:
#thankyou claude
def make_synthetic(ticker, n=2800):
    """Fallback generator matching the stock's profile."""
    rng = np.random.default_rng(42)
    idx = pd.bdate_range(START_DATE, periods=n)
    vol = meta.get("volatility", 0.018)
    base = meta.get("base_price", 100.0)
    r = rng.normal(0.0004, vol, n)
    price = base * np.cumprod(1 + r)
    volume = rng.lognormal(15, 0.3, n) * (1 + 50 * np.abs(r))
    return pd.DataFrame({"Close": price, "Volume": volume}, index=idx)

def load_data(ticker):
    try:
        import yfinance as yf
        print(f"Fetching live market data for {ticker} from yfinance")
        df = yf.download(ticker, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df = df[["Close", "Volume"]].dropna()
        if df.empty or len(df) < 100:
            raise ValueError("Insufficient data returned.")
        print(f"Successfully downloaded {len(df)} trading records for {ticker}.")
        return df, f"{ticker} ({COMPANY_NAME})"
    except Exception as e:
        print(f"Notice: Could not download {ticker} via yfinance ({e}).")
        print("Using realistic market simulation instead.")
        return make_synthetic(ticker), f"{ticker} (SIMULATED: {COMPANY_NAME})"

raw_df, stock_label = load_data(TICKER)
raw_df.tail()


Fetching live market data for TSLA from yfinance
Successfully downloaded 2975 trading records for TSLA.


Price,Close,Volume
Date,,
2026-10-26,391.619995,53090200
2026-10-27,407.459991,45333200
2026-10-28,404.220001,35598800
2026-10-29,403.299988,22858300
2026-10-30,399.809998,41737200


### STEP 4: Economics-Grounded Feature Engineering
using these features derived from eco classes:
1. **`price_change`**: Daily percentage return: $\frac{P_t - P_{t-1}}{P_{t-1}}$.
2. **`quantity_gap`**: Trading volume relative to its 10-day moving average: $\frac{Q_t}{\bar{Q}_{10}} - 1$.
3. **`price_gap`**: Distance of market price from its recent 10-day average: $\frac{P_t}{\bar{P}_{10}} - 1$ 
4. **`excess_demand`**: buyer pressure minus seller pressure:
   $$\text{Excess Demand} = \frac{\sum_{10} V_{\text{up}} - \sum_{10} V_{\text{down}}}{\sum_{10} V_{\text{total}}}$$
   - Positive ($> 0$): **Shortage** (buyers dominate)
   - Negative ($< 0$): **Surplus** (sellers dominate)
5. **`target`**: Tomorrow's percentage price return $\frac{P_{t+1} - P_t}{P_t}$.


In [214]:

def build_dataset(df):
    price = df["Close"]
    quantity = df["Volume"].clip(lower=1)

    out = pd.DataFrame(index=df.index)
    out["close"] = price
    out["price_change"] = price.pct_change()
    out["quantity_gap"] = quantity / quantity.rolling(10).mean() - 1
    out["price_gap"] = price / price.rolling(10).mean() - 1

    #thank you claude
    up_q = quantity.where(out["price_change"] > 0, 0.0)
    down_q = quantity.where(out["price_change"] < 0, 0.0)
    out["excess_demand"] = (up_q.rolling(10).sum() - down_q.rolling(10).sum()) / quantity.rolling(10).sum()

    out["target"] = out["price_change"].shift(-1)
    out = out.dropna()
    next_dates = df.index[df.index.get_indexer(out.index) + 1]
    return out, next_dates

dataset, target_dates = build_dataset(raw_df)
n_total = len(dataset)
split_idx = int(n_total * (1 - TEST_FRAC))
split_date = dataset.index[split_idx]

print(f"Dataset shape: {dataset.shape}")
dataset[["close", "price_change", "quantity_gap", "price_gap", "excess_demand", "target"]].head()


Dataset shape: (2965, 6)


,close,price_change,quantity_gap,price_gap,excess_demand,target
Date,,,,,,
2015-01-15,12.791333,-0.004256,-0.045820,-0.068561,-0.519978,0.006254
2015-01-16,12.871333,0.006254,-0.326615,-0.050642,-0.463924,-0.005905
2015-01-20,12.795333,-0.005905,-0.144586,-0.047744,-0.455112,0.024176
2015-01-21,13.104667,0.024176,-0.178187,-0.017553,-0.515837,0.025691
2015-01-22,13.441333,0.025691,-0.203434,0.012408,-0.367283,-0.001637


### Plot 1: Historical Market Price & 50-Day Moving Average
- **Blue line**: 50-day moving average
- **Narangi region**:test period


In [215]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(dataset.index, dataset["close"], color="grey", lw=1, label="Market Price")
avg50 = dataset["close"].rolling(50).mean()
ax.plot(avg50.index, avg50, color="tab:blue", lw=1.5, label="50-day MA (Equilibrium Level)")
ax.axvspan(split_date, dataset.index[-1], color="orange", alpha=0.15, label="Unseen Test Period")
ax.set_title(f"{stock_label}: Price Trajectory & Equilibrium Trend", fontsize=12)
ax.set_ylabel(f"Price ({CURRENCY})")
ax.legend(loc="upper left")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/01_price_history.png", dpi=150)
plt.show()


### Plot 2: Excess Demand graph
- **Upper Panel**: Closing price over the past n number of  trading days).
- **Lower Panel**: Daily excess demand ratio:
  - 🟢 **Green (> 0)**: Buyers dominate **Shortage** (upward price pressure).
  - 🔴 **Red (< 0)**: Sellers dominate **Surplus** (downward price pressure).


In [216]:
recent = dataset.iloc[-250:]
fig, ax = plt.subplots(2, 1, figsize=(13, 7), sharex=True, gridspec_kw={"height_ratios": [1.3, 1]})
ax[0].plot(recent.index, recent["close"], color="tab:blue", lw=1.3)
ax[0].set_ylabel(f"Price ({CURRENCY})")
ax[0].set_title(f"{stock_label}: Market Price (Past ~1 Year)", fontsize=11)

colors = np.where(recent["excess_demand"] >= 0, "tab:green", "tab:red")
ax[1].bar(recent.index, recent["excess_demand"], color=colors, width=1.0)
ax[1].axhline(0, color="k", lw=0.8)
ax[1].set_ylabel("Excess Demand")
ax[1].set_title("Shortage vs Surplus: Green = Buyers Dominate (Demand > Supply) | Red = Sellers Dominate (Supply > Demand)", fontsize=11)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/02_excess_demand.png", dpi=150)
plt.show()


### Plot 3:Supply & Demand Curve Model
This theoretical diagram showcases the core economics principles:


In [217]:
def plot_microeconomic_model():
    Pd0, Ps0, m, n = 150.0, 50.0, 0.5, 0.5
    Qe = (Pd0 - Ps0) / (m + n)
    Pe = Pd0 - m * Qe
    s = 30.0
    Qe2 = (Pd0 + m * s - Ps0) / (m + n)
    Pe2 = Ps0 + n * Qe2
    cs = 0.5 * Qe * (Pd0 - Pe)
    ps = 0.5 * Qe * (Pe - Ps0)
    ed = (-1 / m) * Pe / Qe
    es = (1 / n) * Pe / Qe #all ts genuinely fried me

    Q = np.linspace(0, 220, 200)
    q1 = np.linspace(0, Qe, 100)
    fig, ax = plt.subplots(figsize=(9.5, 6.5))
    ax.plot(Q, Pd0 - m * Q, color="tab:blue", lw=2, label="Demand curve D ")
    ax.plot(Q, Ps0 + n * Q, color="tab:red", lw=2, label="Supply curve S ")
    ax.plot(Q, Pd0 + m * s - m * Q, color="tab:blue", lw=2, ls="--", label="New Demand D2 ")
    ax.fill_between(q1, Pd0 - m * q1, Pe, color="tab:blue", alpha=0.18, label=f"Consumer Surplus = {cs:.0f}")
    ax.fill_between(q1, Pe, Ps0 + n * q1, color="tab:red", alpha=0.18, label=f"Producer Surplus = {ps:.0f}")
    ax.scatter([Qe], [Pe], color="k", zorder=5)
    ax.annotate(f"Equilibrium E\nP = {Pe:.0f}, Q = {Qe:.0f}", (Qe, Pe), xytext=(Qe + 8, Pe - 28), fontsize=9)
    ax.scatter([Qe2], [Pe2], color="green", zorder=5)
    ax.annotate(f"New Equilibrium E2\nP = {Pe2:.1f}, Q = {Qe2:.0f}\n(Price rises)", (Qe2, Pe2),
                xytext=(Qe2 + 6, Pe2 + 10), fontsize=9, color="green")
    for price, text in [(75, "Shortage (price UP)"), (125, "Surplus : (Price DOWN)")]:
        qs_, qd_ = (price - Ps0) / n, (Pd0 - price) / m
        ax.annotate("", xy=(qd_, price), xytext=(qs_, price), arrowprops=dict(arrowstyle="<->", color="grey"))
        ax.text((qs_ + qd_) / 2, price + 3, text, ha="center", fontsize=9, color="dimgrey")
    ax.text(0.98, 0.04, f"Price Elasticity at E:\nDemand = {ed:.1f}\nSupply = {es:.1f}",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=9,
            bbox=dict(boxstyle="round", fc="white", alpha=0.85))
    ax.set_xlabel("Quantity of Shares Traded (Q)")
    ax.set_ylabel(f"Price (P in {CURRENCY})")
    ax.set_xlim(0, 220)
    ax.set_ylim(0, 170)
    ax.set_title(" Demand, Supply, Surpluses & Elasticity", fontsize=12)
    ax.legend(loc="upper right", fontsize=8.5)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/03_supply_demand_illustration.png", dpi=150)
    plt.show()

plot_microeconomic_model()


### STEP 7: Dataset Preprocessing & 60-Day Sliding Windows
To prevent lookahead bias and shi
85/15


In [218]:
FEATURES = ["price_change", "quantity_gap", "price_gap", "excess_demand"]

scaler_x = StandardScaler().fit(dataset[FEATURES].values[:split_idx])
scaler_y = StandardScaler().fit(dataset[["target"]].values[:split_idx])

X_scaled = scaler_x.transform(dataset[FEATURES].values)
y_scaled = scaler_y.transform(dataset[["target"]].values).ravel()

def make_sequences(X, y, window=LOOKBACK_WINDOW):
    xs, ys = [], []
    for i in range(window, len(X) + 1):
        xs.append(X[i - window:i])
        ys.append(y[i - 1])
    return np.array(xs, dtype=np.float32), np.array(ys, dtype=np.float32)

X_seq, y_seq = make_sequences(X_scaled, y_scaled)
end_row_indices = np.arange(LOOKBACK_WINDOW - 1, n_total)
train_mask = end_row_indices < split_idx

X_train_full, y_train_full = X_seq[train_mask], y_seq[train_mask]
X_test, y_test = X_seq[~train_mask], y_seq[~train_mask]

# Split train_full into train and validation
val_cutoff = int(len(X_train_full) * (1 - VAL_FRAC))
X_train, y_train = X_train_full[:val_cutoff], y_train_full[:val_cutoff]
X_val, y_val = X_train_full[val_cutoff:], y_train_full[val_cutoff:]

print(f"Training sequences   : {X_train.shape} (X) | {y_train.shape} (y)")
print(f"Validation sequences : {X_val.shape} (X)   | {y_val.shape} (y)")
print(f"Testing sequences    : {X_test.shape} (X)  | {y_test.shape} (y)")


Training sequences   : (1966, 60, 4) (X) | (1966,) (y)
Validation sequences : (347, 60, 4) (X)   | (347,) (y)
Testing sequences    : (593, 60, 4) (X)  | (593,) (y)


### STEP 8: LSTM training
The model architecture:
 **2 LSTM Layers** with `hidden_dim = 64` and `dropout = 0.2`.
 **RMSE Tracking**: At every epoch, both **Train RMSE** and **Val RMSE** are printed in standardized and unscaled % units 


In [219]:
class LSTMModel(nn.Module):
    def __init__(self, n_features, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)

def batch_generator(X, y, batch_size=BATCH_SIZE, shuffle=True):
    indices = np.random.permutation(len(X)) if shuffle else np.arange(len(X))
    for i in range(0, len(X), batch_size):
        b_idx = indices[i:i + batch_size]
        yield torch.tensor(X[b_idx]).to(device), torch.tensor(y[b_idx]).to(device)

def predict(model, X):
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in batch_generator(X, np.zeros(len(X), dtype=np.float32), batch_size=BATCH_SIZE, shuffle=False):
            preds.append(model(xb).cpu().numpy())
    return np.concatenate(preds)

def train_lstm(X_tr, y_tr, X_v, y_v, epochs=EPOCHS):
    model = LSTMModel(len(FEATURES)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.MSELoss()
    best_loss = np.inf
    best_weights = None
    patience_counter = 0
    train_rmse_hist, val_rmse_hist = [], []

    target_scale = float(scaler_y.scale_[0])  #conversion to scaled and unscaled ion really get it ngl

    print(f"🚀 Training LSTM for {epochs} epochs (Early Stopping Patience = {PATIENCE})...")
    print("-" * 75)
    print(f"{'Epoch':^8} | {'Train MSE':^11} | {'Val MSE':^11} | {'Train RMSE (%)':^16} | {'Val RMSE (%)':^16}")
    print("-" * 75)

    for epoch in range(1, epochs + 1):
        model.train()
        batch_losses = []
        for xb, yb in batch_generator(X_tr, y_tr, shuffle=True):
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            batch_losses.append(loss.item())

        train_mse = float(np.mean(batch_losses))
        val_preds = predict(model, X_v)
        val_mse = float(np.mean((val_preds - y_v) ** 2))
        
        train_rmse = float(np.sqrt(train_mse))
        val_rmse = float(np.sqrt(val_mse))
        
        train_rmse_hist.append(train_rmse * target_scale * 100)
        val_rmse_hist.append(val_rmse * target_scale * 100)

        if epoch % 5 == 0 or epoch == 1 or epoch == epochs:
            print(f"{epoch:^8d} | {train_mse:^11.4f} | {val_mse:^11.4f} | {train_rmse * target_scale * 100:^16.4f}% | {val_rmse * target_scale * 100:^16.4f}%")

        if val_mse < best_loss:
            best_loss = val_mse
            patience_counter = 0
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"Early stopping at epoch {epoch}: Validation loss stopped improving.")
                break

    print("-" * 75)
    if best_weights is not None:
        model.load_state_dict(best_weights)

    return model, train_rmse_hist, val_rmse_hist

trained_model, tr_rmse_curve, val_rmse_curve = train_lstm(X_train, y_train, X_val, y_val, epochs=EPOCHS)


🚀 Training LSTM for 10 epochs (Early Stopping Patience = 10)...
---------------------------------------------------------------------------
 Epoch   |  Train MSE  |   Val MSE   |  Train RMSE (%)  |   Val RMSE (%)  
---------------------------------------------------------------------------
   1     |   1.0537    |   0.8088    |      3.6423     % |      3.1910     %
   5     |   1.0471    |   0.8091    |      3.6308     % |      3.1916     %
   10    |   1.0466    |   0.8054    |      3.6301     % |      3.1844     %
---------------------------------------------------------------------------


### Plot 4: Training vs Validation Learning Curve (RMSE)
- Displays Root Mean Squared Error (in percentage return units) across all epochs.
- The vertical dashed line marks the checkpoint where validation RMSE reached its minimum (best model weights).
- ~~its not working properly, it should be ignored as of now~~ NOW ITS WORKING


In [220]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.plot(tr_rmse_curve, label="Training RMSE (% return)", color="tab:blue")
ax.plot(val_rmse_curve, label="Validation RMSE (% return)", color="tab:orange")
best_epoch = int(np.argmin(val_rmse_curve))
ax.axvline(best_epoch, color="k", ls="--", lw=0.8, label=f"Best Model (Epoch {best_epoch + 1})")
ax.set_title("LSTM Learning Curve: RMSE across Training Epochs", fontsize=11)
ax.set_xlabel("Epoch")
ax.set_ylabel("RMSE (% Daily Return)")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/04_training_curve.png", dpi=150)
plt.show()


### STEP 9: Out-of-Sample Forecast Evaluation & Test RMSE
We unscale predictions to calculate tomorrow's predicted prices:
$$\hat{P}_{t+1} = P_t \times (1 + \hat{r}_{t+1})$$
And compare them directly against actual market prices $P_{t+1}$ and the **Naive Baseline** ($P_{t+1} = P_t$).


In [221]:
# Inverse transform scaled test predictions
pred_returns = scaler_y.inverse_transform(predict(trained_model, X_test).reshape(-1, 1)).ravel()
actual_returns = scaler_y.inverse_transform(y_test.reshape(-1, 1)).ravel()

test_row_indices = end_row_indices[~train_mask]
test_dates = target_dates[test_row_indices]
today_prices = dataset["close"].values[test_row_indices]

actual_prices = today_prices * (1 + actual_returns)
predicted_prices = today_prices * (1 + pred_returns)
naive_prices = today_prices.copy()

# Test Set RMSE Metrics
rmse = lambda actual, pred: float(np.sqrt(np.mean((actual - pred) ** 2)))
mae = lambda actual, pred: float(np.mean(np.abs(actual - pred)))

return_rmse_lstm = rmse(actual_returns, pred_returns) * 100
return_rmse_naive = rmse(actual_returns, 0) * 100
price_rmse_lstm = rmse(actual_prices, predicted_prices)
price_rmse_naive = rmse(actual_prices, naive_prices)

print("=" * 60)
print(f"TEST SET RMSE EVALUATION: {stock_label}")
print("=" * 60)
print(f"Daily Return RMSE : LSTM {return_rmse_lstm:.4f}% vs Naive {return_rmse_naive:.4f}%")
print(f"Price Level RMSE  : LSTM {CURRENCY}{price_rmse_lstm:.2f} vs Naive {CURRENCY}{price_rmse_naive:.2f}")
print("=" * 60)


TEST SET RMSE EVALUATION: TSLA (Tesla Inc.)
Daily Return RMSE : LSTM 3.7550% vs Naive 3.7547%
Price Level RMSE  : LSTM $12.17 vs Naive $12.15


###  Plot 5: Test Period Forecast vs. Actual Prices
- **Top Panel**: Full test horizon comparing Actual Price (blue) with LSTM Forecast (orange).
- **Bottom Panel**: Zoom on the last 90 trading days comparing Actual vs. LSTM vs. Naive Baseline.


In [222]:
fig, ax = plt.subplots(2, 1, figsize=(13, 9), gridspec_kw={"height_ratios": [1.2, 1]})
train_dates = dataset.index[:split_idx + 1]
ax[0].plot(train_dates, dataset["close"].values[:split_idx + 1], color="grey", lw=1, label="Past Price (Training Period)")
ax[0].plot(test_dates, actual_prices, color="tab:blue", lw=1.3, label="Actual Price (Test Period)")
ax[0].plot(test_dates, predicted_prices, color="tab:orange", lw=1.1, alpha=0.9, label="LSTM Next-Day Forecast")
ax[0].axvline(test_dates[0], color="k", ls=":", lw=1, label="Test Boundary")
ax[0].set_title(f"{stock_label}: Test Period Forecast vs Actual Trajectory", fontsize=12)
ax[0].set_ylabel(f"Price ({CURRENCY})")
ax[0].legend(loc="upper left")

z = min(90, len(test_dates))
ax[1].plot(test_dates[-z:], actual_prices[-z:], "o-", color="tab:blue", ms=3, lw=1.2, label="Actual Price")
ax[1].plot(test_dates[-z:], predicted_prices[-z:], "s-", color="tab:orange", ms=3, lw=1.1, label="LSTM Forecast")
ax[1].plot(test_dates[-z:], naive_prices[-z:], "--", color="green", lw=1, label="Naive Baseline: Tomorrow = Today")
ax[1].set_title(f"Detailed Zoom (Last {z} Trading Days): LSTM vs Naive Baseline", fontsize=11)
ax[1].set_ylabel(f"Price ({CURRENCY})")
ax[1].legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/05_predicted_vs_actual_price.png", dpi=150)
plt.show()


### Plot 6: Forecast Error Distribution & Scatter Plot
- **Left Panel**: Dollar residual error over time ($\hat{P}_t - P_t$).
- **Right Panel**: Scatter of predicted % return vs actual % return against the 45° line of perfect prediction.



In [223]:
#i dont really get this graph tbh, i just asked claude what more should i add. 
err = predicted_prices - actual_prices
hit_rate = np.sign(pred_returns) == np.sign(actual_returns)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
ax[0].plot(test_dates, err, color="tab:purple", lw=0.8)
ax[0].axhline(0, color="k", lw=0.8)
ax[0].set_title(f"Forecast Error: Predicted - Actual ({CURRENCY})", fontsize=11)
ax[0].set_ylabel(f"Error ({CURRENCY})")

ax[1].scatter(pred_returns * 100, actual_returns * 100, s=9, alpha=0.4, color="tab:blue")
lim = max(np.abs(actual_returns).max(), np.abs(pred_returns).max()) * 100
ax[1].plot([-lim, lim], [-lim, lim], "k--", lw=1, label="Perfect Forecast (45° line)")
ax[1].axhline(0, color="grey", lw=0.6)
ax[1].axvline(0, color="grey", lw=0.6)
ax[1].set_xlabel("Predicted Daily Return (%)")
ax[1].set_ylabel("Actual Daily Return (%)")
ax[1].set_title(f"Predicted vs Actual Return (Direction Accuracy: {hit_rate.mean():.1%})", fontsize=11)
ax[1].legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/06_forecast_errors.png", dpi=150)
plt.show()


###  STEP 10: Firm Theory: Backtesting Revenue, Costs & Profit
firm operates only when **Marginal Revenue ($MR$) > Marginal Cost ($MC$)**.
- **Total Revenue ($TR$)**: Gross trading gains.
- **Total Cost ($TC$)**: transaction frictions such as tax and small service cuts ($TC = \text{cost per trade} \times \text{trades}$).
- **Profit ($\pi$)**: $\pi = TR - TC$.


In [224]:
def make_strategy_position(pred_ret, cost_multiplier=2.0, volatility_multiplier=0.15, strategy_mode="active"):
    """Use a cost-aware long/cash rule with a volatility buffer."""
    pred_ret = np.asarray(pred_ret, dtype=float)
    if len(pred_ret) == 0:
        return np.array([], dtype=float), np.array([], dtype=float)
    if strategy_mode == "buy_and_hold":
        return np.ones(len(pred_ret), dtype=float), np.full(len(pred_ret), 0.0)

    recent_vol = np.full(len(pred_ret), np.nan, dtype=float)
    for i in range(len(pred_ret)):
        history = pred_ret[max(0, i - 19):i + 1]
        recent_vol[i] = float(np.std(history, ddof=1)) if len(history) > 1 else 0.0

    entry_threshold = np.maximum(cost_multiplier * TC_RATE, volatility_multiplier * recent_vol)
    position = np.zeros(len(pred_ret), dtype=float)
    for i, forecast in enumerate(pred_ret):
        if i > 0:
            position[i] = position[i - 1]
        if position[i] == 0.0 and forecast > entry_threshold[i]:
            position[i] = 1.0
        elif position[i] == 1.0 and forecast <= 0.0:
            position[i] = 0.0
    return position, entry_threshold


def select_strategy_params(pred_ret, true_ret):
    candidates = [("active", c, v) for c in (1.0, 2.0, 3.0, 4.0) for v in (0.0, 0.1, 0.2, 0.3)]
    candidates.append(("buy_and_hold", 0.0, 0.0))
    best_score, best_params = -np.inf, candidates[0]
    for strategy_mode, cost_multiplier, volatility_multiplier in candidates:
        position, _ = make_strategy_position(pred_ret, cost_multiplier, volatility_multiplier, strategy_mode)
        trades = np.abs(np.diff(position, prepend=0.0))
        wealth = CAPITAL + np.cumsum(CAPITAL * position * true_ret - CAPITAL * TC_RATE * trades)
        peak = np.maximum.accumulate(wealth)
        drawdown = float(np.max((peak - wealth) / peak))
        score = float(wealth[-1]) - CAPITAL * drawdown
        if score > best_score:
            best_score, best_params = score, (strategy_mode, cost_multiplier, volatility_multiplier)
    return {"strategy_mode": best_params[0], "cost_multiplier": best_params[1], "volatility_multiplier": best_params[2]}


def backtest_strategy(dates, actual_ret, pred_ret, strategy_params=None):
    strategy_params = strategy_params or {}
    position, entry_threshold = make_strategy_position(
        pred_ret, strategy_params.get("cost_multiplier", 2.0),
        strategy_params.get("volatility_multiplier", 0.15),
        strategy_params.get("strategy_mode", "active")
    )
    trade_signals = np.abs(np.diff(position, prepend=0.0))
    daily_gross = CAPITAL * position * actual_ret
    daily_cost = CAPITAL * TC_RATE * trade_signals

    def summarize(gross, cost, trades):
        tr = float(gross.sum())
        tc = float(cost.sum())
        return {
            "tr": tr, "tc": tc, "profit": tr - tc, "trades": int(trades.sum()),
            "cum_tr": np.cumsum(gross), "cum_tc": np.cumsum(cost),
            "wealth": CAPITAL + np.cumsum(gross - cost)
        }

    strat = summarize(daily_gross, daily_cost, trade_signals)
    bh_trades = np.zeros_like(actual_ret)
    bh_trades[0] = 1
    bh = summarize(CAPITAL * actual_ret, CAPITAL * TC_RATE * bh_trades, bh_trades)

    # Plot 7
    fig, ax = plt.subplots(3, 1, figsize=(13, 11), sharex=True, gridspec_kw={"height_ratios": [1.3, 1.3, 0.6]})
    ax[0].plot(dates, strat["cum_tr"], color="tab:green", lw=1.6, label="Total Revenue TR (Gross Gains)")
    ax[0].plot(dates, strat["cum_tc"], color="tab:red", lw=1.6, label="Total Cost TC (Trading Frictions)")
    ax[0].fill_between(dates, strat["cum_tc"], strat["cum_tr"], where=strat["cum_tr"] >= strat["cum_tc"],
                        color="green", alpha=0.15, label="Profit (TR > TC)")
    ax[0].fill_between(dates, strat["cum_tc"], strat["cum_tr"], where=strat["cum_tr"] < strat["cum_tc"],
                        color="red", alpha=0.15, label="Loss (TR < TC)")
    ax[0].axhline(0, color="k", lw=0.6)
    ax[0].set_title("Trading Strategy as a Firm: Total Revenue vs Total Cost (Profit = TR - TC)", fontsize=12)
    ax[0].set_ylabel(f"Amount ({CURRENCY})")
    ax[0].legend(fontsize=8.5)

    ax[1].plot(dates, strat["wealth"], color="tab:orange", lw=1.6, label="LSTM Active Strategy")
    ax[1].plot(dates, bh["wealth"], color="tab:blue", lw=1.6, label="Buy & Hold (Passive Benchmark)")
    ax[1].axhline(CAPITAL, color="k", ls="--", lw=0.8, label="Initial Capital")
    ax[1].set_title(f"Cumulative Wealth from {CURRENCY}{CAPITAL:,} during Test Horizon", fontsize=12)
    ax[1].set_ylabel(f"Wealth ({CURRENCY})")
    ax[1].legend(fontsize=8.5)

    ax[2].fill_between(dates, position, 0, step="post", color="tab:orange", alpha=0.6)
    ax[2].set_yticks([0, 1])
    ax[2].set_yticklabels(["Cash", "Hold Stock"])
    ax[2].set_title(f"Cost-aware Allocation ({strat['trades']} trades, entry threshold averages {entry_threshold.mean() * 100:.2f}%)", fontsize=11)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/07_revenue_cost_profit.png", dpi=150)
    plt.show()

    strat["strategy"] = "Validation-adaptive LSTM (long/cash)"
    strat["entry_threshold_pct"] = round(float(entry_threshold.mean()) * 100, 4)
    return strat, bh

val_pred_returns = scaler_y.inverse_transform(predict(trained_model, X_val).reshape(-1, 1)).ravel()
val_actual_returns = scaler_y.inverse_transform(y_val.reshape(-1, 1)).ravel()
strategy_params = select_strategy_params(val_pred_returns, val_actual_returns)
print("Selected validation strategy parameters:", strategy_params)
strat_results, bh_results = backtest_strategy(test_dates, actual_returns, pred_returns, strategy_params)


Selected validation strategy parameters: {'strategy_mode': 'active', 'cost_multiplier': 2.0, 'volatility_multiplier': 0.0}


### STEP 11: Investor Utility Theory & Diminishing Marginal Utility
- **Utility Function**: $$U(W) = \sqrt{\frac{W}{C}}$$
- **Marginal Utility**: $$MU(W) = \frac{dU}{dW} = \frac{1}{2\sqrt{W \cdot C}} > 0, \quad \frac{d^2U}{dW^2} < 0$$
- Concavity explains **risk aversion**: losing an amount destroys more utility than gaining the same amount creates.


In [225]:
def plot_utility_analysis(w_start, w_strat, w_bh):
    u = lambda w: np.sqrt(np.asarray(w) / SCALE_W)
    mu = lambda w: 0.5 / np.sqrt(np.asarray(w) / SCALE_W)

    pts = [("Initial Capital", w_start, "k"), ("LSTM Strategy", w_strat, "tab:orange"), ("Buy & Hold", w_bh, "tab:blue")]
    lo = min(0.6 * w_start, 0.9 * min(w_strat, w_bh))
    hi = 1.15 * max(w_start, w_strat, w_bh)
    W = np.linspace(lo, hi, 300)

    step = 0.1 * w_start
    gain_u = float(u(w_start + step) - u(w_start))
    loss_u = float(u(w_start) - u(w_start - step))

    fig, ax = plt.subplots(2, 1, figsize=(10, 8.5), sharex=True)
    ax[0].plot(W, u(W), color="tab:green", lw=2)
    for name, w, c in pts:
        ax[0].scatter([w], [float(u(w))], color=c, s=70, zorder=5,
                      label=f"{name}: {CURRENCY}{w:,.0f}, TU = {float(u(w)):.3f}")
    ax[0].set_ylabel("Total Utility TU (utils)")
    ax[0].set_title("Total Utility: Concave Shape Models Risk Aversion", fontsize=11)
    ax[0].legend(fontsize=8.5, loc="lower right")

    ax[1].plot(W, mu(W), color="tab:purple", lw=2)
    for name, w, c in pts:
        ax[1].scatter([w], [float(mu(w))], color=c, s=70, zorder=5)
    ax[1].set_xlabel(f"Wealth W ({CURRENCY})")
    ax[1].set_ylabel("Marginal Utility MU (utils/wealth unit)")
    ax[1].set_title("Diminishing Marginal Utility: Incremental Utility Decreases with Wealth", fontsize=11)
    ax[1].text(0.98, 0.95,
               f"+{CURRENCY}{step:,.0f} gain adds {gain_u:.4f} utils\n-{CURRENCY}{step:,.0f} loss destroys {loss_u:.4f} utils\n"
               "Key Insight: Downside risk hurts more than equal upside rewards!",
               transform=ax[1].transAxes, ha="right", va="top", fontsize=9,
               bbox=dict(boxstyle="round", fc="white", alpha=0.85))
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/08_utility.png", dpi=150)
    plt.show()

    delta_u_strat = float(u(w_strat) - u(w_start))
    delta_u_bh = float(u(w_bh) - u(w_start))
    return delta_u_strat, delta_u_bh

du_strat, du_bh = plot_utility_analysis(CAPITAL, strat_results["wealth"][-1], bh_results["wealth"][-1])


### STEP 12:Conclusions


In [226]:
hit_dir = float(np.mean(np.sign(pred_returns) == np.sign(actual_returns)))
up_ratio = float(np.mean(actual_returns > 0))

train_data = dataset.iloc[:split_idx]
shortage_ret = train_data.loc[train_data["excess_demand"] > 0, "target"].mean() * 100
surplus_ret = train_data.loc[train_data["excess_demand"] < 0, "target"].mean() * 100

print("=" * 76)
print(f"ECONOMICS PBL REPORT: {stock_label} (Test Days: {len(actual_returns)})")
print("=" * 76)
print("1) DEMAND AND SUPPLY DYNAMICS:")
print(f"   • Next-day return following SHORTAGE : {shortage_ret:+.3f}%")
print(f"   • Next-day return following SURPLUS  : {surplus_ret:+.3f}%")
if shortage_ret > surplus_ret:
    print("   • Conclusion: Price adjustments adhere to the Law of Supply and Demand.")
else:
    print("   • Conclusion: Rapid market arbitrage balances short-term volume imbalances quickly.")

print("\n2) MACHINE LEARNING REGRESSION BENCHMARK (RMSE & MAE):")
print(f"   • Return RMSE: LSTM {return_rmse_lstm:.4f}% vs Naive {return_rmse_naive:.4f}%")
print(f"   • Return MAE : LSTM {mae(actual_returns, pred_returns)*100:.4f}% vs Naive {mae(actual_returns, 0)*100:.4f}%")
print(f"   • Price RMSE : LSTM {CURRENCY}{price_rmse_lstm:.2f} vs Naive {CURRENCY}{price_rmse_naive:.2f}")
print(f"   • Directional Accuracy: {hit_dir:.1%} (Market upward rate: {up_ratio:.1%})")

print(f"\n3) FIRM ECONOMICS: Revenue, Costs & Profit (Initial Capital: {CURRENCY}{CAPITAL:,}):")
print(f"   {'Strategy':18s}{'Revenue':>12s}{'Cost':>10s}{'Profit':>12s}{'Trades':>8s}{'Final Wealth':>16s}")
for nm, d in [("LSTM Strategy", strat_results), ("Buy & Hold", bh_results)]:
    print(f"   {nm:18s}{CURRENCY}{d['tr']:>11,.0f}{CURRENCY}{d['tc']:>9,.0f}{CURRENCY}{d['profit']:>11,.0f}{d['trades']:>8d}"
          f"{CURRENCY}{d['wealth'][-1]:>15,.0f}")
if strat_results["trades"] > 0:
    print(f"   • Average Revenue per Trade : {CURRENCY}{strat_results['tr'] / strat_results['trades']:,.0f}")
    print(f"   • Marginal Cost per Trade   : {CURRENCY}{CAPITAL * TC_RATE:,.0f}")

print(f"\n4) INVESTOR UTILITY ANALYSIS:")
print(f"   • Net Utility Change (LSTM Strategy) : {du_strat:+.4f} utils")
print(f"   • Net Utility Change (Buy & Hold)    : {du_bh:+.4f} utils")
print("   • Diminishing MU explains why steady, lower-volatility returns are preferred by risk-averse people.")
print("=" * 76)

# Walk-forward classification: market-relative 5/10-day targets, LSTM, Random Forest, regimes, and risk metrics.
from strategy import run_walk_forward, benchmark_ticker
from plots import plot_walk_forward_diagnostics
from data import load_data
benchmark_symbol = benchmark_ticker(TICKER)
benchmark_df, _, _ = load_data(benchmark_symbol, START_DATE, END_DATE)
walk_forward_results = {str(h): run_walk_forward(raw_df, benchmark_df, CAPITAL, TC_RATE, EPOCHS, h) for h in (5, 10)}
for horizon, result in walk_forward_results.items():
    print(f"\n{horizon}-day market-relative target")
    report = pd.DataFrame(result["walk_forward"]).T
    percentage_columns = ["cagr", "volatility", "max_drawdown", "win_rate", "turnover"]
    currency_columns = ["final_wealth", "profit", "transaction_cost"]
    format_map = {column: "{:,.2f}" for column in report.columns}
    format_map.update({column: "{:,.2f}%" for column in percentage_columns if column in report})
    format_map.update({column: "{:,.2f}" for column in currency_columns if column in report})
    display(report.style.format(format_map, na_rep="-").set_caption(f"{horizon}-day market-relative performance"))
    diagnostics = result.get("_diagnostics", [])
    if diagnostics:
        plot_walk_forward_diagnostics(diagnostics, result["walk_forward"], CURRENCY, f"{OUT_DIR}/{horizon}_day_diagnostics.png")
    else:
        print("Diagnostics are unavailable in this result. Rerun this cell to refresh the walk-forward results.")


ECONOMICS PBL REPORT: TSLA (Tesla Inc.) (Test Days: 593)
1) DEMAND AND SUPPLY DYNAMICS:
   • Next-day return following SHORTAGE : +0.271%
   • Next-day return following SURPLUS  : +0.067%
   • Conclusion: Price adjustments adhere to the Law of Supply and Demand.

2) MACHINE LEARNING REGRESSION BENCHMARK (RMSE & MAE):
   • Return RMSE: LSTM 3.7550% vs Naive 3.7547%
   • Return MAE : LSTM 2.7260% vs Naive 2.7245%
   • Price RMSE : LSTM $12.17 vs Naive $12.15
   • Directional Accuracy: 50.1% (Market upward rate: 51.3%)

3) FIRM ECONOMICS: Revenue, Costs & Profit (Initial Capital: $100,000):
   Strategy               Revenue      Cost      Profit  Trades    Final Wealth
   LSTM Strategy     $     70,052$    4,700$     65,352      47$        165,352
   Buy & Hold        $    120,290$      100$    120,190       1$        220,190
   • Average Revenue per Trade : $1,490
   • Marginal Cost per Trade   : $100

4) INVESTOR UTILITY ANALYSIS:
   • Net Utility Change (LSTM Strategy) : +0.2859 utils


,final_wealth,profit,transaction_cost,trades,cagr,volatility,sharpe,sortino,max_drawdown,win_rate,turnover,folds
lstm,"735,117.67","635,117.67","4,000.00",40.00,0.47%,0.52%,0.98,0.75,0.86%,0.09%,0.03%,3.00
random_forest,"1,727,956.92","1,627,956.92","11,300.00",113.00,0.73%,0.59%,1.22,1.03,0.87%,0.11%,0.09%,3.00
buy_and_hold,"1,411,323.93","1,311,323.93",100.00,1.00,0.66%,1.32%,1.03,1.87,1.00%,0.49%,0.00%,3.00
momentum,"7,059,359.85","6,959,359.85","13,100.00",131.00,1.26%,0.93%,1.33,1.89,0.87%,0.26%,0.10%,3.00


Diagnostics are unavailable in this result. Rerun this cell to refresh the walk-forward results.

10-day market-relative target


,final_wealth,profit,transaction_cost,trades,cagr,volatility,sharpe,sortino,max_drawdown,win_rate,turnover,folds
lstm,"656,507.63","556,507.63","3,600.00",36.00,0.44%,0.73%,0.84,0.60,0.97%,0.09%,0.03%,3.00
random_forest,"515,734.78","415,734.78","8,800.00",88.00,0.37%,0.84%,0.78,0.57,0.98%,0.10%,0.07%,3.00
buy_and_hold,"16,656,726.61","16,556,726.61",100.00,1.00,1.67%,1.94%,1.44,2.75,1.00%,0.51%,0.00%,3.00
momentum,"814,767,287.41","814,667,287.41","12,800.00",128.00,4.64%,1.39%,1.90,3.01,0.99%,0.27%,0.10%,3.00


Diagnostics are unavailable in this result. Rerun this cell to refresh the walk-forward results.
